# Capacitação Ciência de Dados — Semana 5
## Regressão Linear, Regressão Logística e Séries Temporais

**Dataset:** Superstore Sales (Kaggle)  
**Período:** 2015 – 2018  
**Registros:** ~10.000 vendas de uma loja de varejo americana

In [ ]:
# Instala o Prophet caso ainda não esteja disponível no ambiente
# As demais bibliotecas já fazem parte da distribuição padrão do Anaconda/pip
# !pip install prophet --quiet

## 1. Introdução

### Sobre o dataset

O **Superstore Sales** é um dataset público disponível no Kaggle que simula as operações de uma loja de varejo americana entre 2015 e 2018. Ele contém informações sobre pedidos, clientes, produtos, descontos, vendas e lucros.

### Problemas propostos

| Modelo | Pergunta de negócio | Variável alvo |
|--------|--------------------|--------------|
| Regressão Linear | Quanto uma venda vai gerar em receita? | `Sales` (contínua) |
| Regressão Logística | Essa venda vai dar lucro ou prejuízo? | `Profit_flag` (binária) |
| Séries Temporais | Qual será a receita diária nos próximos 3 meses? | `Sales` agregado por dia |

### Justificativa das escolhas

- **Regressão linear:** `Sales` é uma variável contínua e positiva, influenciada por desconto, quantidade e categoria — relação razoavelmente linear após transformações.
- **Regressão logística:** Binarizar o lucro (`Profit > 0`) cria um problema de classificação relevante para o negócio: identificar vendas com risco de prejuízo antes de fechá-las.
- **Prophet:** O dataset cobre 4 anos de vendas diárias com sazonalidade semanal e anual visível, cenário ideal para o Prophet da Meta.

In [ ]:
# --- Imports gerais ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente em todo o notebook
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

print('Bibliotecas carregadas com sucesso.')

In [ ]:
# Carregameto do dataset — ajuste o caminho se necessário
# Arquivo original do Kaggle: 'Sample - Superstore.csv'
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print(f'Shape: {df.shape}')
df.head()

---
## 2. Análise Exploratória (EDA)

In [ ]:
# Visão geral dos tipos e valores ausentes
print('=== Tipos de dados ===')
print(df.dtypes)
print('\n=== Valores ausentes ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df[['Sales', 'Profit', 'Discount', 'Quantity']].describe().round(2)

In [ ]:
# Conversão da coluna de data — necessária para a série temporal
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False)

print(f'Período: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total de pedidos únicos: {df["Order ID"].nunique()}')

In [ ]:
# --- Distribuição de Sales (faturamento bruto) e Profit (distribuição de lucros) ---
fig, axes = plt.subplots(1, 2)

# Aplicamos log1p para reduzir o efeito dos outliers na visualização
axes[0].hist(np.log1p(df['Sales']), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de Sales (log1p)')
axes[0].set_xlabel('log(1 + Sales)')

axes[1].hist(df['Profit'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribuição de Profit')
axes[1].set_xlabel('Profit (USD)')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1, label='zero')
axes[1].legend()

plt.suptitle('Distribuições das variáveis alvo', y=1.02)
plt.tight_layout()
plt.show()

# Observação: Sales tem distribuição muito assimétrica — usaremos log1p como feature. >> para fins estabilizadores da variância e evitar divisões incorretas
# Profit apresenta vendas com prejuízo (< 0), o que motiva a criação do Profit_flag.

In [ ]:
# --- Relação entre desconto e lucro ---
plt.figure()
sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', alpha=0.4, s=20
)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Desconto vs Lucro por categoria')
plt.xlabel('Discount')
plt.ylabel('Profit (USD)')
plt.tight_layout()
plt.show()

# Observação: descontos acima de ~0.4 estão fortemente associados a prejuízo,
# o que torna Discount um preditor importante para a regressão logística.

In [ ]:
# --- Matriz de correlação (variáveis numéricas) ---> resumo das relações entre variáveis
corr = df[['Sales', 'Profit', 'Discount', 'Quantity']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Matriz de correlação')
plt.tight_layout()
plt.show()

In [ ]:
# --- Detecção de outliers em Sales (IQR) ---> voulme de transações que fogem do padrão 
Q1, Q3 = df['Sales'].quantile(0.25), df['Sales'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (df['Sales'] < Q1 - 1.5 * IQR) | (df['Sales'] > Q3 + 1.5 * IQR)

print(f'Outliers em Sales: {outlier_mask.sum()} registros ({outlier_mask.mean():.1%} do total)')

# Estratégia: manteremos os outliers no conjunto completo pois são vendas legítimas
# (ex: equipamentos de escritório de alto valor). Na regressão usaremos log1p(Sales)
# para reduzir o impacto desses valores extremos.

### Justificativa das variáveis preditoras

| Feature | Tipo | Justificativa |
|---------|------|---------------|
| `Discount` | Numérica | Alta correlação negativa com Profit; impacta Sales indiretamente |
| `Quantity` | Numérica | Volume vendido influencia receita diretamente |
| `Category` | Categórica (OHE) | Três categorias com perfis de margem distintos |
| `Region` | Categórica (OHE) | Regiões com pricing e demanda diferente |
| `Segment` | Categórica (OHE) | Consumer, Corporate, Home Office têm tickets médios diferentes |